# 01 — Import battery-cycling data
This notebook answers the first question in a gregorithm workflow: *what measurements were retained in canonical scientific form?* It uses a tiny public synthetic CSV and finishes in seconds after the notebook dependencies are installed. Time is expressed in seconds, current in amperes, and voltage in volts; positive current means charge and negative current means discharge. The source is synthetic, and the optional local path remains disabled unless you explicitly change it.


## Choose the input
Choose **Run All** to use the checked-in example. To inspect your own already-canonical CSV, change only LOCAL_FILE; keep private files outside this project. Ambiguous column names or units require an explicit ColumnMapping — never guess units or reverse current automatically.


In [ ]:
import tempfile
from pathlib import Path


def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples" / "data" / "synthetic_cycling.csv").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the gregorithm repository folder.")


PROJECT_ROOT = find_project_root()
LOCAL_FILE = None  # Optional: Path(r"D:/your-local/canonical-file.csv")
SOURCE = (
    Path(LOCAL_FILE)
    if LOCAL_FILE is not None
    else PROJECT_ROOT / "examples" / "data" / "synthetic_cycling.csv"
)
_session = tempfile.TemporaryDirectory(prefix="gregorithm-notebook-")
WORK = Path(_session.name)
print(f"Input: {SOURCE.name}")
print(f"Temporary result area: {WORK}")

In [ ]:
from gregorithm import inspect_source

inspection = inspect_source(SOURCE)
print(f"Source kind: {inspection.kind.value}")
print(f"Columns: {inspection.columns}")
print(f"Mapping confirmed automatically: {inspection.mapping_confirmed}")

In [ ]:
from gregorithm import ImportRequest, import_source

result = import_source(
    ImportRequest(SOURCE, WORK / "import", "synthetic-cell", "import-job", "import-v1")
)
report = result.report
print(f"Source rows: {report.source_rows}")
print(f"Retained rows: {report.retained_rows}")
print(f"Invalid I/V rows removed: {report.removed_invalid_measurement}")
print(f"Duplicate timestamps removed: {report.removed_duplicate_time}")
print(f"Canonical content hash: {report.content_hash}")

In [ ]:
import pyarrow.parquet as pq

canonical = pq.read_table(result.ref.path / "points" / "canonical.parquet")
print(canonical.schema)
canonical.slice(0, 8).to_pylist()

The canonical table is the scientific input for later analyses. The three measurements preserve the retained real data in canonical units; the row position is the source row used by later analyses. The content hash identifies retained scientific content rather than the original filename.


## What happens to an invalid measurement?
The next cell creates a second synthetic file with one non-finite current. Import removes that row, records an import diagnostic, and does not propagate a quality flag into later scientific calculations.


In [ ]:
invalid_source = WORK / "synthetic_with_invalid_row.csv"
invalid_source.write_text(
    "time_s,current_a,voltage_v\n0,0,3.60\n1,nan,3.61\n2,1,3.62\n", encoding="utf-8"
)
invalid_result = import_source(
    ImportRequest(invalid_source, WORK / "invalid-import", "invalid-demo", "job", "generation")
)
print(f"Retained rows: {invalid_result.report.retained_rows}")
print(f"Removed invalid measurements: {invalid_result.report.removed_invalid_measurement}")
print([diagnostic.code.value for diagnostic in invalid_result.report.diagnostics])

A removed row is absent from the canonical table. An unavailable value must never be interpreted as numerical zero. Continue with notebook 02 to identify cycles and calculate Capacity.
